In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.stats import norm
from linearmodels.iv import IV2SLS

### Unobserved Confounders & Instrumental Variables

#### The Problem: Unobserved Confounders

What happens when the confounder is **unobserved**? For example:

| Scenario | Unobserved Confounder |
|----------|----------------------|
| Education → Income | **Natural ability** (affects both education and income) |
| Ad Campaign → Sales | **Customer personality** (affects both ad engagement and purchase behavior) |

**Challenge:** If you can't measure the confounder, you can't control for it with regression or PSM.


#### The Solution: Instrumental Variables (IV)

An **Instrumental Variable (IV)** is a variable that:

| Condition | Description |
|-----------|-------------|
| **1. Causes the treatment (X)** | IV affects whether someone receives the treatment |
| **2. Does NOT cause the outcome (Y) except through X** | IV has no direct effect on Y |
| **3. Is random** | As good as a coin flip |

> **An IV is like a "natural experiment." It randomly pushes some people into treatment and others into control, mimicking an RCT.**

In [2]:
# Scenario: Does education (X) increase income (Y)?
# Unobserved confounder: "Natural Ability" (Z)
# Ability causes both education AND income!

np.random.seed(42)
n = 1000

# Unobserved confounder (we CAN'T measure this)
natural_ability = np.random.normal(0, 1, n)

# Education (depends on ability)
education = 12 + 2 * natural_ability + np.random.normal(0, 0.5, n)

# Income (depends on education AND ability)
income = 30 + 2 * education + 5 * natural_ability + np.random.normal(0, 3, n)

df = pd.DataFrame({
    'Education': education,
    'Income': income,
    # We DON'T have 'Natural_Ability' in our data!
})

# --- OLS Regression (Confounded!) ---
model = LinearRegression()
model.fit(df[['Education']], df['Income'])
print("OLS Regression (Confounded by unobserved Ability):")
print(f"  Education coefficient: {model.coef_[0]:.4f}")
print(f"  True effect: 2.0000")
print(f"  Problem: We're overestimating because ability causes both!\n")

# The true effect is 2.0, but OLS gives ~4.0 because ability is a confounder

OLS Regression (Confounded by unobserved Ability):
  Education coefficient: 4.3972
  True effect: 2.0000
  Problem: We're overestimating because ability causes both!



### What is a Good Instrument?

Think of an Instrumental Variable (IV) as a **"natural randomization"** that affects your treatment but has **NO independent effect** on your outcome.

#### Example 1: Distance to College

| Component | Variable |
|-----------|----------|
| **IV** | Distance from home to the nearest college |
| **Treatment (X)** | Years of education |
| **Outcome (Y)** | Income |

**Why it works:**
- Living close to college makes it **cheaper/easier** to go to school → affects education
- Living close to college does **NOT** directly make you richer → only affects income through education

#### Example 2: Rainfall

| Component | Variable |
|-----------|----------|
| **IV** | Rainfall in a region |
| **Treatment (X)** | Mobile phone ownership |
| **Outcome (Y)** | GDP growth |

**Why it works:**
- More rain → better crops → more people can afford phones
- Rain does **NOT** directly affect GDP → only through economic activity

#### Example 3: Lottery Draft

| Component | Variable |
|-----------|----------|
| **IV** | Being drafted (random assignment) |
| **Treatment (X)** | Military service |
| **Outcome (Y)** | Lifetime earnings |

**Why it works:**
- Draft lottery is **truly random** (like a coin flip)
- Being drafted causes military service
- Being drafted does **NOT** directly affect earnings → only through military service

In [6]:
# Generate data with an Instrumental Variable
np.random.seed(42)
n = 1000

# 1. The Instrument (Z) - A random "natural experiment"
# Example: Distance to college (randomly assigned by geography)
distance_to_college = np.random.uniform(0, 10, n)

# 2. Unobserved confounder (Ability)
ability = np.random.normal(0, 1, n)

# 3. Treatment (Education)
# Depends on distance (IV) AND ability (confounder)
education = 12 + 0.5 * distance_to_college + ability + np.random.normal(0, 0.5, n)

# 4. Outcome (Income)
# Depends on education AND ability
income = 30 + 2 * education + 2 * ability + np.random.normal(0, 3, n)

df = pd.DataFrame({
    'Education': education,
    'Income': income,
    'Distance': distance_to_college,
    # 'Ability' is UNOBSERVED (we don't have it in the data)
})

print("="*60)
print("DATA")
print("="*60)
print(df)

# --- Stage 1: OLS (Confounded) ---
ols_model = LinearRegression()
ols_model.fit(df[['Education']], df['Income'])
ols_effect = ols_model.coef_[0]

print("="*60)
print("INSTRUMENTAL VARIABLES (2SLS) DEMONSTRATION")
print("="*60)
print(f"OLS Effect (Confounded): {ols_effect:.4f}")
print(f"True Effect: 2.0000\n")

# --- Stage 2: Two-Stage Least Squares (2SLS) ---

# Stage 1: Regress Treatment (Education) on Instrument (Distance)
stage1_model = LinearRegression()
stage1_model.fit(df[['Distance']], df['Education'])

# Get predicted values (the part of education caused by distance)
df['Education_Predicted'] = stage1_model.predict(df[['Distance']])

# Stage 2: Regress Outcome (Income) on Predicted Education
stage2_model = LinearRegression()
stage2_model.fit(df[['Education_Predicted']], df['Income'])
iv_effect = stage2_model.coef_[0]

print(f"IV Effect (2SLS): {iv_effect:.4f}")
print(f"  This is much closer to the true effect (2.0)!")

# --- Manual IV Calculation (The Formula) ---
# IV Effect = Cov(Z, Y) / Cov(Z, X)
cov_zy = np.cov(df['Distance'], df['Income'])[0, 1]
cov_zx = np.cov(df['Distance'], df['Education'])[0, 1]
iv_formula = cov_zy / cov_zx

print(f"\nManual IV Calculation:")
print(f"  Cov(Z, Y) / Cov(Z, X) = {iv_formula:.4f}")
print(f"  Matches the 2SLS result!")

DATA
     Education     Income  Distance
0    13.347243  59.673439  3.745401
1    15.376674  56.133364  9.507143
2    15.287807  57.726408  7.319939
3    15.983906  60.062851  5.986585
4    13.381104  56.420179  1.560186
..         ...        ...       ...
995  10.995450  50.765482  0.915821
996  15.846709  54.976445  9.173136
997  12.526308  51.293210  1.368186
998  16.291002  55.496636  9.502374
999  13.777329  55.911606  4.460058

[1000 rows x 3 columns]
INSTRUMENTAL VARIABLES (2SLS) DEMONSTRATION
OLS Effect (Confounded): 2.4757
True Effect: 2.0000

IV Effect (2SLS): 1.8599
  This is much closer to the true effect (2.0)!

Manual IV Calculation:
  Cov(Z, Y) / Cov(Z, X) = 1.8599
  Matches the 2SLS result!


### The Three Requirements for a Valid Instrument

| Requirement | What it means | How to test it |
|-------------|---------------|----------------|
| **1. Relevance** | $Z$ must be correlated with $X$ | F-statistic > 10 (in first stage) |
| **2. Exogeneity** | $Z$ must be independent of confounders | Domain knowledge (it must be "as good as random") |
| **3. Exclusion Restriction** | $Z$ must only affect $Y$ through $X$ | Domain knowledge (no direct effect) |

In [8]:
# 1. Relevance (The F-statistic)
# Always check the first-stage F-statistic, not the overall F, to validate your instrument!

# Check if instrument is strong enough
stage1_model = LinearRegression()
stage1_model.fit(df[['Distance']], df['Education'])

# F-statistic (we want > 10 for a strong instrument)
r2 = stage1_model.score(df[['Distance']], df['Education'])
f_stat = (r2 / (1 - r2)) * (len(df) - 2) / 1

print(f"F-statistic for Distance in Stage 1: {f_stat:.2f}")
print(f"  {'✅ Strong instrument (> 10)' if f_stat > 10 else '⚠️ Weak instrument'}")

F-statistic for Distance in Stage 1: 1754.52
  ✅ Strong instrument (> 10)


In [ ]:
#Linear (regression) models for Python. Extends statsmodels with Panel regression, 
#instrumental variable estimators, system estimators and models for estimating asset prices:

In [18]:
# Formula: y ~ exog + [endog ~ instruments]
mod = IV2SLS.from_formula('Income ~ 1 + [Education ~ Distance]', data=df)
results = mod.fit(cov_type='robust') # default is robust
print(results)

# Get the F-statistic for each endogenous variable
first_stage = results.first_stage
f_statistic = first_stage.diagnostics['f.stat']
p_value = first_stage.diagnostics['f.pval']

print(f"First-stage F-statistic: {f_statistic.iloc[0]:.2f}")
print(f"p-value: {p_value.iloc[0]:.4f}")

                          IV-2SLS Estimation Summary                          
Dep. Variable:                 Income   R-squared:                      0.5814
Estimator:                    IV-2SLS   Adj. R-squared:                 0.5810
No. Observations:                1000   F-statistic:                    497.85
Date:                Sun, Aug 09 2026   P-value (F-stat)                0.0000
Time:                        16:10:54   Distribution:                  chi2(1)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept      32.202     1.2192     26.412     0.0000      29.812      34.591
Education      1.8599     0.0834     22.313     0.00

#### Model Overview

| Metric | Value | Interpretation |
|--------|-------|----------------|
| **Dep. Variable** | Income | The outcome we're trying to explain |
| **Estimator** | IV-2SLS | Instrumental Variables — Two-Stage Least Squares |
| **No. Observations** | 1,000 | Sample size |
| **R-squared** | 0.5814 | The model explains **58.1%** of the variation in Income |
| **Adj. R-squared** | 0.5810 | Adjusted for degrees of freedom (virtually the same) |
| **F-statistic** | 497.85 | Overall model is **highly significant** |
| **P-value (F-stat)** | 0.0000 | Strong evidence that the model fits better than random |


#### Parameter Estimates

| Variable | Coefficient | Std. Error | T-stat | P-value | 95% CI |
|----------|-------------|------------|--------|---------|--------|
| **Intercept** | 32.20 | 1.22 | 26.41 | 0.0000 | [29.81, 34.59] |
| **Education** | **1.86** | 0.083 | 22.31 | 0.0000 | [1.70, 2.02] |

**Interpretation:**

> **1 additional year of education → 1,860 increase in income** (measured in 1000s)


#### Key Metrics

| Statistic | Value | What It Means |
|-----------|-------|---------------|
| **First-stage F-statistic** | **1751.18** | The instrument (Distance) is **very strong** (> 10) ✅ |
| **P-value (First-stage)** | 0.0000 | Highly significant — Distance strongly predicts Education |
| **Endogenous** | Education | The variable we're treating as potentially confounded |
| **Instrument** | Distance | Our "natural experiment" that affects Education but not Income directly |
| **Cov. Estimator** | Robust | Standard errors are corrected for heteroskedasticity |


#### Comparing OLS vs IV

| Method | Education Coefficient | Interpretation |
|--------|-----------------------|----------------|
| **OLS (Confounded)** | 2.48 (from earlier) | Overestimates due to ability bias ❌ |
| **IV (Causal)** | **1.86** | True causal effect — ability bias removed ✅ |
| **True Effect** | 2.00 | The ground truth used to generate the data |

**Difference:** OLS overestimated by **~$480** due to unobserved ability (confounding).


In [21]:
# OLS Overestimate = OLS - True = 2.4757 - 2.0000 = 0.4757 × $1000 = $480 per year
# IV Underestimate = IV - True = 1.8599 - 2.0000 = -0.1401 × $1000 = $140 per year